The dataset represents a collection of text extracted from the works of William Shakespeare. It contains a variety of plays, sonnets and his other literary works. The text is in English and includes a wide range of vocabulary and linguistic styles typical of Shakespearean literature.

In [9]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import random
import torch.optim as optim

### 2. Import the shakespeare dataset

In [3]:
with open("input.txt", "r", encoding="utf-8") as f:
    text = f.read()

print(text[:500])
print(f"\nDataset length: {len(text)} characters")

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citizens.

First Citizen:
We are accounted poor

Dataset length: 1115394 characters


### 3. Prepare text data for training
A character based model capable of predicting the next character in a sequence based on the preceding characters. Character-based models work at finer granularity, handling individual characters. The following code start by creating character mappings and sequences.

In [4]:
# Creating character mappings and sequences
chars = sorted(list(set(text)))
chars_indices = dict((c, i) for i, c in enumerate(chars))
indices_char = dict((i, c) for i, c in enumerate(chars))

# Creating sequences
maxlen = 40
step = 3
sentences = []
next_chars = []
for i in range(0, len(text) - maxlen, step):
    sentences.append(text[i: i + maxlen])
    next_chars.append(text[i + maxlen])
print("Number of sequences:", len(sentences))

Number of sequences: 371785


### 4. Vectorize the input sequence into binary matrices
representing the presence of characters in the vocabulary. In addition encode target outputs as one-hot vectors indicating the next character in the sequence.


In [6]:
# Vectorizing the data
X = torch.zeros((len(sentences), maxlen, len(chars)), dtype=torch.float32)
y = torch.zeros((len(sentences), len(chars)), dtype=torch.float32)
for i, sentence in enumerate(sentences):
    for t, char in enumerate(sentence):
        X[i, t, chars_indices[char]] = 1
    y[i, chars_indices[next_chars[i]]] = 1


### 5. Import the required libraries and build a simple RNN model

In [7]:
# Buidling the model
class RNNModel(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super(RNNModel, self).__init__()
        self.rnn = nn.RNN(input_size, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, output_size)
        self.softmax = nn.Softmax(dim=1)

    def forward(self, x):
        out, _ = self.rnn(x)
        out = self.fc(out[:, -1, :])
        out = self.softmax(out)
        return out

In [8]:
input_size = len(chars)
hidden_size = 128
output_size = len(chars)

model = RNNModel(input_size, hidden_size, output_size)

In [10]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

### 6. Train the model for 10 epochs

In [11]:
# Training the model
num_epochs = 10
batch_size = 128
for epoch in range(num_epochs):
    for i in range(0, len(X), batch_size):
        X_batch = X[i:i+batch_size]
        y_batch = y[i:i+batch_size]

        # Forward pass
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)

        # Backward pass and optimization
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    print(f'Each [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}')

Each [1/10], Loss: 4.1183
Each [2/10], Loss: 4.1183
Each [3/10], Loss: 4.1183
Each [4/10], Loss: 4.1183
Each [5/10], Loss: 4.1183
Each [6/10], Loss: 4.1183
Each [7/10], Loss: 4.1183
Each [8/10], Loss: 4.1183
Each [9/10], Loss: 4.1183
Each [10/10], Loss: 4.1183


### 10. Create a function which should generate text with the specified word count (100 words) using a trained language model

In [15]:
import numpy as np

def generate_text(model, seed_text, char_to_idx, idx_to_char,
                  seq_length, word_count=100, temperature=1.0):
    """
    Generate text using a trained character-level language model.

    Parameters:
        model        : Trained Keras model
        seed_text    : Starting text
        char_to_idx  : Dictionary mapping characters to indices
        idx_to_char  : Dictionary mapping indices to characters
        seq_length   : Length of input sequences used during training
        word_count   : Number of words to generate
        temperature  : Controls randomness

    Returns:
        Generated text containing approximately word_count words.
    """

    vocab_size = len(char_to_idx)
    generated = seed_text

    while len(generated.split()) < word_count:

        # Get the last seq_length characters
        sequence = generated[-seq_length:]

        # Pad if necessary
        if len(sequence) < seq_length:
            sequence = " " * (seq_length - len(sequence)) + sequence

        # One-hot encode the sequence
        x = np.zeros((1, seq_length, vocab_size), dtype=np.float32)

        for t, char in enumerate(sequence):
            if char in char_to_idx:
                x[0, t, char_to_idx[char]] = 1

        # Predict probabilities
        preds = model.predict(x, verbose=0)[0]

        # Apply temperature sampling
        preds = np.asarray(preds).astype("float64")
        preds = np.log(preds + 1e-8) / temperature
        exp_preds = np.exp(preds)
        preds = exp_preds / np.sum(exp_preds)

        # Sample next character
        next_index = np.random.choice(range(vocab_size), p=preds)
        next_char = idx_to_char[next_index]

        # Append character
        generated += next_char

    # Trim to exactly 100 words
    generated = " ".join(generated.split()[:word_count])

    return generated